## Step 2

Inputs: 
Output: New

This creates a new column in the 

In [ ]:
import pandas as pd
import numpy as np
import awswrangler as wr
import time

# Add timing and logging
print("Starting data load...")
start_time = time.time()

# Read all Parquet files from the S3 folder - UNCOMMENT THIS
s3_path = "s3://thesis--ec331-s3/FCAS_RAISE1SEC-Volume-Bids/PUBLIC_DVD_BIDPEROFFER1_202311010000.parquet/"
df = wr.s3.read_parquet(path=s3_path)

print(f"Data loaded in {time.time() - start_time:.2f} seconds")
print(f"DataFrame shape: {df.shape}")
print(f"Memory usage: {df.memory_usage().sum() / 1024 / 1024:.2f} MB")

# Print some sample data to understand structure
print("\nSample data:")
sample = df.sample(3)
print(sample[["DUID", "TRADINGDATE", "PERIODID", "MAXAVAIL"] + 
      [f"PRICEBAND{i}" for i in range(1, 11)] + 
      [f"BANDAVAIL{i}" for i in range(1, 11)]])

# Define a truly vectorized approach for capping
def cap_bands_vectorized(df_group):
    """Process a group of bids with the same DUID, TRADINGDATE, PERIODID"""
    # Create a copy to avoid modifying the original
    result_df = df_group.copy()
    
    # For each row in the group
    for idx, row in result_df.iterrows():
        max_avail = row["MAXAVAIL"]
        
        # Extract prices and volumes
        prices = [row[f"PRICEBAND{i}"] if f"PRICEBAND{i}" in row and not pd.isna(row[f"PRICEBAND{i}"]) else np.nan 
                 for i in range(1, 11)]
        volumes = [row[f"BANDAVAIL{i}"] if f"BANDAVAIL{i}" in row and not pd.isna(row[f"BANDAVAIL{i}"]) else 0.0 
                  for i in range(1, 11)]
        
        # Create a list of (price, volume, band_index)
        bands = [(p, v, i) for i, (p, v) in enumerate(zip(prices, volumes), 1) if not pd.isna(p)]
        
        # Sort by price
        bands.sort(key=lambda x: x[0])
        
        # Apply capping
        running_sum = 0.0
        capped_volumes = [0.0] * 10
        
        for price, volume, band_idx in bands:
            if running_sum >= max_avail:
                break
                
            remaining = max_avail - running_sum
            capped_volume = min(volume, remaining)
            
            capped_volumes[band_idx-1] = capped_volume
            running_sum += capped_volume
        
        # Update the result dataframe
        for i in range(1, 11):
            col = f"BANDAVAIL{i}"
            if col in result_df.columns:
                result_df.at[idx, col] = capped_volumes[i-1]
    
    return result_df

# Much more efficient approach
def process_data(df, chunk_size=1000):
    print("\nProcessing data in chunks...")
    capped_dfs = []
    total_chunks = (len(df) + chunk_size - 1) // chunk_size
    
    for i in range(0, len(df), chunk_size):
        chunk_start = time.time()
        print(f"Processing chunk {i//chunk_size + 1}/{total_chunks}")
        
        # Get chunk
        chunk = df.iloc[i:i+chunk_size].copy()
        
        # Process chunk - group by key columns
        grouped = chunk.groupby(["DUID", "TRADINGDATE", "PERIODID"])
        capped_chunk = grouped.apply(cap_bands_vectorized)
        
        capped_dfs.append(capped_chunk)
        print(f"Chunk {i//chunk_size + 1} completed in {time.time() - chunk_start:.2f} seconds")
    
    # Combine results
    if capped_dfs:
        return pd.concat(capped_dfs)
    else:
        return pd.DataFrame()

# Execute the processing
capped_df = process_data(df, chunk_size=500)
print(f"\nAll processing completed in {time.time() - start_time:.2f} seconds")
print(capped_df.head())